In [10]:
# %%
# Notebook 08: 08_eval_full_dataset.ipynb
# هدف: اجرای کامل گراف روی structured_questions.csv و محاسبه Accuracy کلی و per-category

import os, sys, time, json, traceback
from pathlib import Path

import pandas as pd
from tqdm import tqdm

PROJECT_ROOT = Path.cwd().parent  # اگر داخل notebooks هستی
SRC_PATH = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_PATH))

from dotenv import load_dotenv
load_dotenv()

print(f"✓ PROJECT_ROOT: {PROJECT_ROOT}")
print(f"✓ SRC_PATH: {SRC_PATH}")
print("✓ OPENROUTER_API_KEY:", (os.getenv("OPENROUTER_API_KEY") or "")[:10], "...")
print("✓ COHERE_API_KEY:", (os.getenv("COHERE_API_KEY") or "")[:10], "...")


✓ PROJECT_ROOT: f:\Thesis\project\3-Multi-Agent-System
✓ SRC_PATH: f:\Thesis\project\3-Multi-Agent-System\src
✓ OPENROUTER_API_KEY: sk-or-v1-1 ...
✓ COHERE_API_KEY: O0pIM8Zkm9 ...


In [11]:
# %%
from legal_multi_agent.graphs.main_graph import graph

print("✓ Graph imported")

✓ Graph imported


In [12]:
# %%
QUESTIONS_PATH = r"F:\Thesis\project\403-vekalat\structured_questions.csv"
GOLD_PATH      = r"F:\Thesis\project\1-BaselineTest\GOLD\Gold.csv"

df_q = pd.read_csv(QUESTIONS_PATH)
df_gold = pd.read_csv(GOLD_PATH)

print("Questions:", len(df_q))
print("Gold:", len(df_gold))

display(df_q.head(2))
display(df_gold.head(2))


Questions: 119
Gold: 119


,question_number,category,question,options
0,1,حقوق مدنی,کدام یک از موارد زیر صحیح است؟,1) با توافق طرفین امکان از بین بردن سبب انفساخ...
1,2,حقوق مدنی,شخص «الف» حین رانندگی با خودرو سواری با شخص «ب...,1) دیه هر دو راننده صرفاً تا میزان دیه کامل و ...


,idx,Gold
0,1,3
1,2,2


In [13]:
# %%
# حذف سوال 89
if "idx" in df_gold.columns and 89 in df_gold["idx"].values:
    df_gold = df_gold[df_gold["idx"] != 89].reset_index(drop=True)

multi_gold = {
    90: ["3", "4"],
    53: ["1", "2"],
    52: ["2", "4"],
    48: ["1", "3"],
    46: ["1", "3"],
    36: ["1", "4"],
    30: ["1", "2"],
    25: ["3", "4"],
    9:  ["1", "2"],
    4:  ["1", "4"],
    3:  ["2", "4"],
}

def normalize_gold_value(idx: int, gold_val: str) -> str:
    if idx in multi_gold:
        return ",".join(multi_gold[idx])

    val = str(gold_val).strip()
    digits = [ch for ch in val if ch.isdigit()]
    if not digits:
        return val
    if len(digits) == 1:
        return digits[0]
    return ",".join(digits)

df_gold["Gold"] = [
    normalize_gold_value(int(idx), gold)
    for idx, gold in zip(df_gold["idx"], df_gold["Gold"])
]

display(df_gold.head(5))


,idx,Gold
0,1,3
1,2,2
2,3,"2,4"
3,4,"1,4"
4,5,3


In [14]:
# %%
def parse_options_to_text(options_raw) -> str:
    """تبدیل ستون options به متن استاندارد برای گراف."""
    if options_raw is None or (isinstance(options_raw, float) and pd.isna(options_raw)):
        return ""
    s = str(options_raw).strip()

    # اگر JSON-لایک بود
    if s.startswith("[") or s.startswith("{"):
        try:
            obj = json.loads(s)
            if isinstance(obj, list):
                lines = []
                for i, item in enumerate(obj, start=1):
                    lines.append(f"{i}) {str(item).strip()}")
                return "\n".join(lines)
        except Exception:
            pass

    return s

def is_correct(pred: str, gold: str) -> bool:
    gold_set = {g.strip() for g in str(gold).split(",") if g.strip()}
    return str(pred).strip() in gold_set


In [15]:
# %%
df = df_q.copy()

# اطمینان از اسم ستون‌ها
assert "question_number" in df.columns
assert "question" in df.columns
assert "options" in df.columns

assert "idx" in df_gold.columns
assert "Gold" in df_gold.columns

df = df.merge(df_gold[["idx", "Gold"]], left_on="question_number", right_on="idx", how="left")
df.drop(columns=["idx"], inplace=True)

print("Merged rows:", len(df))
print("Gold missing:", df["Gold"].isna().mean())
display(df.head(3))


Merged rows: 119
Gold missing: 0.0


,question_number,category,question,options,Gold
0,1,حقوق مدنی,کدام یک از موارد زیر صحیح است؟,1) با توافق طرفین امکان از بین بردن سبب انفساخ...,3
1,2,حقوق مدنی,شخص «الف» حین رانندگی با خودرو سواری با شخص «ب...,1) دیه هر دو راننده صرفاً تا میزان دیه کامل و ...,2
2,3,حقوق مدنی,کدام مورد در خصوص تصرفی که همراه با قصد تملک ب...,1) تصرف در مواردی مملک است حتی اگر همراه با قص...,"2,4"


In [16]:
# %%
USE_OPTION_VERIFIER = True
USE_RETRIEVER_TOOL  = True

MAX_REVISIONS = 2
SLEEP_SECONDS = 0.2  

OUT_PATH = PROJECT_ROOT / "data" / "eval" / "notebook08_full_v2_results.csv"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Output will be saved to:", OUT_PATH)


Output will be saved to: f:\Thesis\project\3-Multi-Agent-System\data\eval\notebook08_full_v2_results.csv


In [17]:
from tqdm import tqdm 

def run_one(row) -> dict:
    qnum = int(row["question_number"])

    question = str(row["question"])
    options_text = parse_options_to_text(row["options"])

    state = {
        "question_number": qnum,
        "question": question,
        "options_text": options_text,
        "max_revisions": MAX_REVISIONS,
        "use_option_verifier": USE_OPTION_VERIFIER,
        "use_retriever_tool": USE_RETRIEVER_TOOL,
        "messages": [],
        "tool_results": {},
        "verifier_output": None,
        "revision_count": 0,
    }

    t0 = time.time()
    try:
        result = graph.invoke(state)
        elapsed = time.time() - t0

        final = result.get("final_toon") or {}
        pred = final.get("answer")
        conf = final.get("confidence")
        expl = final.get("explanation")

        tqdm.write(f"[Q{qnum}] pred={pred} | conf={conf}")


        # ✅ این بخش اضافه شده: چاپ پاسخ انتخابی مدل
        gold = row.get("Gold")
        ok = (is_correct(pred, gold) if pd.notna(gold) and pred else None)

        critic = result.get("critic_toon") or {}
        critic_needs = critic.get("needs_revision")

        verifier = result.get("verifier_output") or {}
        v_rec = verifier.get("recommended_answer")
        v_conf = verifier.get("confidence")

        ctx = result.get("context") or ""
        rag_results = result.get("rag_results") or []

        return {
            "question_number": qnum,
            "pred": pred,
            "confidence": conf,
            "explanation": expl,
            "gold": gold,
            "is_correct": ok,
            "critic_needs_revision": critic_needs,
            "revision_count": result.get("revision_count"),
            "verifier_recommended": v_rec,
            "verifier_confidence": v_conf,
            "context_len": len(ctx),
            "n_docs": len(rag_results),
            "elapsed_sec": elapsed,
            "error": None,
        }

    except Exception as e:
        elapsed = time.time() - t0

        # ✅ چاپ خطا برای همان سؤال
        tqdm.write(f"[Q{qnum}] ERROR: {type(e).__name__}: {str(e)[:200]}")

        return {
            "question_number": qnum,
            "pred": None,
            "confidence": None,
            "explanation": None,
            "gold": row.get("Gold"),
            "is_correct": None,
            "critic_needs_revision": None,
            "revision_count": None,
            "verifier_recommended": None,
            "verifier_confidence": None,
            "context_len": None,
            "n_docs": None,
            "elapsed_sec": elapsed,
            "error": f"{type(e).__name__}: {str(e)}",
        }


In [18]:
# %%
done_set = set()
if OUT_PATH.exists():
    df_done = pd.read_csv(OUT_PATH)
    done_set = set(df_done["question_number"].astype(int).tolist())
    print("Resuming. Already done:", len(done_set))

rows_out = []
if OUT_PATH.exists():
    rows_out = df_done.to_dict("records")

for _, row in tqdm(df.iterrows(), total=len(df)):
    qnum = int(row["question_number"])
    if qnum in done_set:
        continue

    r = run_one(row)
    rows_out.append(r)
    done_set.add(qnum)

    # ذخیره دوره‌ای
    if len(rows_out) % 10 == 0:
        pd.DataFrame(rows_out).to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

    time.sleep(SLEEP_SECONDS)

df_res_v2 = pd.DataFrame(rows_out)
df_res_v2.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print("Saved to:", OUT_PATH)
print("Rows in result:", len(df_res_v2))


  0%|          | 0/119 [00:00<?, ?it/s]

📝 Query: کدام یک از موارد زیر صحیح است؟

گزینه‌ها:
1) با توافق طرفین امکان از بین بردن سب...
🔍 روش انتخاب شده: Semantic search
🔍 Semantic only: 24 نتیجه
   ✓ Retrieved: 24 documents
🔄 Reranking 24 سند (بدون law+article)...
   ✓ Reranked: top 6 documents


  0%|          | 0/119 [00:22<?, ?it/s]

   ⚠️ [Critic] needs_revision=true but no valid citation → forcing false
[Q1] pred=3 | conf=5


  1%|          | 1/119 [00:22<43:53, 22.32s/it]

...

📝 Query: در قانون اساسی به تعیین تکلیف کدام یک از امور مربوط به هیئت منصفه در جرایم سیاسی...
🎯 روش انتخاب شده: Metadata-aware
🎯 Metadata: law='قانون اساسی' | None None
🔍 فیلتر فقط قانون...
   ✓ 10 سند (law_only)
🔍 Semantic fallback...
✅ نهایی: 24 نتیجه
   ✓ Retrieved: 24 documents
🔄 Reranking 24 سند (بدون law+article)...
   ✓ Reranked: top 6 documents


 99%|█████████▉| 118/119 [46:55<00:17, 17.27s/it]

   ⚠️ [Critic] needs_revision=true but no valid citation → forcing false
[Q140] pred=4 | conf=5


100%|██████████| 119/119 [46:55<00:00, 23.66s/it]

Saved to: f:\Thesis\project\3-Multi-Agent-System\data\eval\notebook08_full_v2_results.csv
Rows in result: 119


In [23]:
df_res_v2["category"] = df_q["category"]

In [27]:
# %%
overall_acc = df_res_v2["is_correct"].mean()
coverage = df_res_v2["pred"].notna().mean()
error_rate = df_res_v2["error"].notna().mean()

print("Overall accuracy (v2):", overall_acc)
print("Coverage (has pred):", coverage)
print("Error rate:", error_rate)

print(df_res_v2.groupby("category")["is_correct"].mean())

out_path = PROJECT_ROOT / "data" / "eval" / "notebook08_full_v2_results.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
df_res_v2.to_csv(out_path, index=False, encoding="utf-8-sig")
print("Saved to:", out_path)


Overall accuracy (v2): 0.6302521008403361
Coverage (has pred): 1.0
Error rate: 0.0
category
آیین دادرسی مدنی     0.600000
آیین دادرسی کیفری    0.750000
حقوق اساسی           0.700000
حقوق تجارت           0.750000
حقوق جزا             0.315789
حقوق مدنی            0.650000
Name: is_correct, dtype: float64
Saved to: f:\Thesis\project\3-Multi-Agent-System\data\eval\notebook08_full_v2_results.csv


In [28]:
# %%
agg = df_res_v2.groupby("category").agg(
    n=("question_number", "count"),
    acc=("is_correct", "mean"),
    coverage=("pred", lambda s: s.notna().mean()),
    error_rate=("error", lambda s: s.notna().mean()),
    avg_time=("elapsed_sec", "mean"),
)
display(agg.sort_values("n", ascending=False))


,n,acc,coverage,error_rate,avg_time
category,,,,,
آیین دادرسی مدنی,20,0.600000,1.0,0.0,18.782998
آیین دادرسی کیفری,20,0.750000,1.0,0.0,20.743720
حقوق اساسی,20,0.700000,1.0,0.0,22.470579
حقوق تجارت,20,0.750000,1.0,0.0,33.556428
حقوق مدنی,20,0.650000,1.0,0.0,20.802327
حقوق جزا,19,0.315789,1.0,0.0,24.394658
